# TRACER — Module 8: Explainable AI (XAI)

The capstone of the AI engine: runs the real pipeline (detect → attribute → reconstruct) on
one real image, then generates a real, human-readable forensic explanation tying Modules 5,
6, and 7 together — reusing the trained detector saved to Drive in Module 5, so no
retraining needed.

In [ ]:
GITHUB_USERNAME = "IqraYasmin123"   # change if this isn't your username
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/tracers.git"

import os
if not os.path.exists("/content/tracers"):
    !git clone {REPO_URL} /content/tracers
else:
    %cd /content/tracers
    !git pull

%cd /content/tracers/ai-engine
import sys
sys.path.insert(0, ".")

from src.vlm import CLIPVLM, VLMConfig
from src.dataset import FlickrDataset, DatasetConfig
from src.attacks import PGDAttack, AttackConfig
from src.detection import EntropyClassifier, extract_attention_entropy
from src.attribution import GradientSaliency
from src.reconstruction import DiffusionReconstructor, ReconstructionConfig, compute_reconstruction_metrics
from src.xai import ExplanationContext, RuleBasedExplainer
import torch
import numpy as np
print("Imports OK.")


## Log into Hugging Face (recommended)

In [ ]:
from huggingface_hub import login
HF_TOKEN = ""  # paste your token here
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in.")
else:
    print("No token provided — continuing without login (may hit rate limits).")


## Load CLIP, Flickr8k, and the trained detector from Module 5

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

vlm = CLIPVLM(VLMConfig())
print(f"CLIP loaded on {vlm.device}")

DRIVE_DATASET_DIR = "/content/drive/MyDrive/TRACER/datasets/flickr8k"
LOCAL_DATASET_DIR = "/content/flickr8k_local"
COPY_MARKER = os.path.join(LOCAL_DATASET_DIR, ".copy_complete")

if not os.path.exists(COPY_MARKER):
    print("Copying Flickr8k to local disk...")
    !rm -rf "{LOCAL_DATASET_DIR}"
    !cp -r "{DRIVE_DATASET_DIR}" "{LOCAL_DATASET_DIR}"
    with open(COPY_MARKER, "w") as f:
        f.write("done")
else:
    print("Local copy already present and verified complete.")

dataset = FlickrDataset(DatasetConfig(root_dir=LOCAL_DATASET_DIR, image_size=(224, 224)))
print(f"Loaded {len(dataset)} (image, caption) pairs.")

DETECTOR_PATH = "/content/drive/MyDrive/TRACER/models/entropy_detector.joblib"
if os.path.exists(DETECTOR_PATH):
    detector = EntropyClassifier().load(DETECTOR_PATH)
    print("Loaded trained detector from Module 5.")
else:
    print("No saved detector found — run Module 5's notebook first to train and save one.")


## Run the real pipeline: attack, detect, attribute, reconstruct

In [ ]:
sample = dataset[0]
real_image = sample["image"]
true_caption = sample["caption"]

pixel_values = vlm.preprocess_image(real_image)
text_embeds = vlm.encode_text([true_caption])
pgd_adv = PGDAttack(AttackConfig(epsilon=8/255, alpha=2/255, steps=10)).generate(
    vlm, pixel_values, text_embeds
)

# --- Detect ---
entropy_features = extract_attention_entropy(vlm, pgd_adv).cpu().numpy()
detection_probs = detector.predict_proba(entropy_features)[0]
predicted_label = detector.predict(entropy_features)[0]
detection_confidence = detection_probs[list(detector.classes_).index(predicted_label)]
print(f"Detection: {predicted_label} ({detection_confidence:.1%} confidence)")

# --- Attribute ---
attribution = GradientSaliency().generate(vlm, pgd_adv, text_embeds)
peak_fraction = float((attribution.heatmap > 0.5).mean())
print(f"Attribution: {attribution.method}, {peak_fraction:.1%} of image above threshold")


## Reconstruct and gather fidelity metrics

In [ ]:
def to_pil(pixel_values, processor, device):
    mean = torch.tensor(processor.image_processor.image_mean).view(1, 3, 1, 1).to(device)
    std = torch.tensor(processor.image_processor.image_std).view(1, 3, 1, 1).to(device)
    img = (pixel_values * std + mean).clamp(0, 1)
    arr = (img.squeeze().permute(1, 2, 0).detach().cpu().numpy() * 255).astype("uint8")
    from PIL import Image
    return Image.fromarray(arr)

adversarial_pil = to_pil(pgd_adv, vlm.processor, vlm.device)

from diffusers import StableDiffusionInpaintPipeline
pipeline = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16
).to(vlm.device)

reconstructor = DiffusionReconstructor(config=ReconstructionConfig(mask_threshold=0.6), pipeline=pipeline)
heatmap_for_mask = attribution.resized((adversarial_pil.height, adversarial_pil.width))
reconstructed = reconstructor.reconstruct(adversarial_pil, heatmap_for_mask)

recon_metrics = compute_reconstruction_metrics(real_image, reconstructed)
baseline_metrics = compute_reconstruction_metrics(real_image, adversarial_pil)
print(f"Reconstruction SSIM: {recon_metrics['SSIM']:.4f}, PSNR: {recon_metrics['PSNR']:.2f} dB")
print(f"Baseline (no reconstruction) SSIM: {baseline_metrics['SSIM']:.4f}")


## Build the explanation context and generate the real report

This is the actual "AI reasoning summary" — everything above condensed into a human-readable
forensic explanation.

In [ ]:
context = ExplanationContext(
    detection_verdict=predicted_label,
    detection_confidence=float(detection_confidence),
    attribution_method=attribution.method,
    attribution_peak_fraction=peak_fraction,
    reconstruction_ssim=recon_metrics["SSIM"],
    reconstruction_psnr=recon_metrics["PSNR"],
    baseline_ssim=baseline_metrics["SSIM"],
)

explanation = RuleBasedExplainer().explain(context)
print(explanation.full_text())


## Module 8 completion check

Run the full test suite:
```bash
cd ai-engine
pytest tests/test_xai.py -v
```
All 20 tests should pass. Combined with the real explanation generated above — read it and
check that every sentence accurately reflects the real numbers computed earlier in this
notebook — Module 8 is complete.

**This closes out the entire AI engine (Modules 2-8).** Every core forensic capability from
your abstract — passive detection, gradient/attention-based attribution, diffusion
reconstruction, and human-readable explanation — has been built, unit tested, and verified
against real data. Continue to **Module 9 — FastAPI Backend**, which starts the second half
of the project: exposing this AI engine as a real web API.